# Attention Foundations — Google Colab runner

Build and test the project's basic attention implementations using the same Python files as local development. This initial runner checks the boilerplate; attention mechanisms remain pending.

Upload this notebook through File → Upload notebook in Google Colab. Connect to a CPU runtime, run in order, and upload both attention-foundations-source.zip and source-manifest.json at the first prompt. Generate them using src/export_colab.py from the project checkout. Download the result archive before ending the runtime.

The kernel coordinates a separate locked Python 3.12.13 environment with CPU PyTorch. No Drive mount or public repository is required. Installation needs internet. Colab storage is temporary: https://research.google.com/colaboratory/faq.html


## 1. Upload and verify the source

Select the source ZIP and its matching manifest together. Every member hash and path must validate before extraction. Hash agreement checks consistency of your source pair, not the identity of an unknown sender.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import io
import json
import subprocess
import sys
import tempfile
import zipfile
from google.colab import files

uploaded = files.upload()
archive_name = "attention-foundations-source.zip"
manifest_name = "source-manifest.json"
if set(uploaded) != {archive_name, manifest_name}:
    raise ValueError("Select exactly the source ZIP and its matching source-manifest.json together.")
manifest = json.loads(uploaded[manifest_name])
if manifest.get("schema_version") != 1 or manifest.get("project_id") != "attention-foundations":
    raise ValueError("Upload an attention-foundations schema 1 source manifest.")
archive_bytes = uploaded[archive_name]
archive_hash = hashlib.sha256(archive_bytes).hexdigest()
if archive_hash != manifest["archive_sha256"]:
    raise ValueError("ZIP hash mismatch. Export/upload a matching ZIP and manifest.")

verified = {}
prefix = "attention-foundations/"
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for entry in archive.infolist():
        name = entry.filename
        relative = name.removeprefix(prefix)
        parts = PurePosixPath(relative).parts
        if (not name.startswith(prefix) or entry.is_dir() or not parts
                or relative.startswith("/") or ".." in parts or "\\" in name
                or relative != PurePosixPath(relative).as_posix() or relative in verified):
            raise ValueError(f"Unexpected or duplicate ZIP path: {name}")
        payload = archive.read(entry)
        if hashlib.sha256(payload).hexdigest() != manifest["files"].get(relative):
            raise ValueError(f"Source file hash mismatch: {relative}")
        verified[relative] = payload
if set(verified) != set(manifest["files"]):
    raise ValueError("ZIP members do not match the manifest.")

WORK = Path(tempfile.mkdtemp(prefix="attention-foundations-colab-", dir="/content"))
SOURCE = WORK / "attention-foundations"
for relative, payload in verified.items():
    target = SOURCE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
RESULTS = WORK / "results"
RESULTS.mkdir()
(RESULTS / manifest_name).write_bytes(uploaded[manifest_name])
print("Source directory:", SOURCE)
print("Verified source files:", len(verified))
print("Archive SHA-256:", archive_hash)
print("Source revision:", manifest["source_revision"])
print("Includes local source changes:", manifest["working_tree"])


## 2. Install the locked environment

Install pinned uv into a tools environment, then synchronize the source lock. This uses the project Python rather than the Colab kernel. Installation can take several minutes.


In [ ]:
import os

def run(*args):
    (RESULTS / "receipt.json").unlink(missing_ok=True)
    environment = os.environ.copy()
    environment.pop("UV_PROJECT_ENVIRONMENT", None)
    environment["UV_LINK_MODE"] = "copy"
    completed = subprocess.run([str(arg) for arg in args], cwd=SOURCE, env=environment,
                               text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    number = len(list(RESULTS.glob("command-*.log"))) + 1
    (RESULTS / f"command-{number:02d}.log").write_text(completed.stdout)
    print(completed.stdout)
    completed.check_returncode()
    return completed.stdout

TOOLS = WORK / "tools"
run(sys.executable, "-m", "venv", TOOLS)
run(TOOLS / "bin/python", "-m", "pip", "install", "uv==0.10.10")
UV = TOOLS / "bin/uv"
run(UV, "sync", "--frozen", "--python", "3.12.13")
PYTHON = SOURCE / ".venv/bin/python"


## 3. Record actual environment versions

Keep this environment record with the source manifest. Selecting a GPU does not change this CPU dependency lock.


In [ ]:
environment_text = run(PYTHON, "-c", """
import json, platform, torch, numpy
print(json.dumps({"python": platform.python_version(), "torch": torch.__version__,
                  "numpy": numpy.__version__, "platform": platform.platform(),
                  "execution": "CPU correctness profile"}))
""")
environment = json.loads(environment_text)
(RESULTS / "environment.json").write_text(json.dumps(environment, indent=2))


## 4. Check the boilerplate

The small CPU square/gradient probe checks dependency setup. It does not run an attention mechanism or a translation model.


In [ ]:
status = json.loads(run(PYTHON, "src/attention_foundations.py", "status"))
setup = json.loads(run(PYTHON, "src/check_setup.py"))
(RESULTS / "status.json").write_text(json.dumps(status, indent=2))
(RESULTS / "setup.json").write_text(json.dumps(setup, indent=2))
assert status["status"] == "boilerplate" and status["implemented_mechanisms"] == []
assert setup["setup_checks"]["autograd_probe"] == "passed"


## 5. Run the project tests

Failures, skips and an empty suite stop before a success receipt. Rerunning any command invalidates an older receipt.


In [ ]:
import xml.etree.ElementTree as ET

run(PYTHON, "-m", "pytest", "-q", "--junitxml", RESULTS / "pytest.xml")
suites = list(ET.parse(RESULTS / "pytest.xml").getroot().iter("testsuite"))
totals = {key: sum(int(suite.get(key, "0")) for suite in suites)
          for key in ("tests", "failures", "errors", "skipped")}
if totals["tests"] <= 0 or any(totals[key] for key in ("failures", "errors", "skipped")):
    raise RuntimeError("Need a nonempty suite with no failures, errors or skips")
receipt = {"project": "attention-foundations", "scope": "boilerplate/setup checks",
           "archive_sha256": archive_hash, "source_revision": manifest["source_revision"],
           "working_tree": manifest["working_tree"], "environment": environment,
           "pytest": totals, "implemented_mechanisms": [], "passed": True}
(RESULTS / "receipt.json").write_text(json.dumps(receipt, indent=2))
print(json.dumps(receipt, indent=2))


## 6. Download the evidence

Save source identity, actual versions, status/setup results, command logs, the JUnit report and receipt. To use a fresh runtime later, upload the same source pair and rerun from the top.


In [ ]:
import shutil

receipt_path = RESULTS / "receipt.json"
if not receipt_path.is_file() or not json.loads(receipt_path.read_text()).get("passed"):
    raise RuntimeError("Run setup and every test successfully before downloading a success receipt")
archive = shutil.make_archive(str(WORK / "attention-foundations-colab-results"), "zip", RESULTS)
files.download(archive)
print("Retain the result ZIP before deleting this runtime. Mechanism implementation remains pending.")
